# Normalización de nombres

En esta sección se realiza una limpieza de los datos con el fin de corregir los errores detectados en la sección de Análisis Exploratorio de Datos. <br/> Se corrigen los errores en los identificadores del dataset y se renombran los archivos que contienen espacios o que no siguen un formato establecido.

In [34]:
from vasijas.config import load_config
from vasijas.paths import ProjectPaths
from vasijas.eda import (
    agregar_marca_identificador_en_fs,
    cargar_metadata,
    contar_identificadores_encontrados,
    contar_identificadores_no_encontrados,
    contar_identificadores_unicos_no_encontrados,
    corregir_identificadores,
    get_archivos_sin_registro,
    get_identificadores_metadata,
    get_identificadores_no_encontrados,
    get_nombres_archivos_perfiles,
)
from vasijas.files import (
    copiar_png_normalizados_por_ubicacion,
    guardar_metadata,
    normalizar_identificadores,
)

In [35]:
CONFIG = load_config()
PATHS = ProjectPaths.from_root(CONFIG.project_root)

## Normalización de los nombres de los archivos

Se aplican las siguentes reglas a los nombres de los archivos:
- Deben estar escritos en minúsculas.
- No deben tener acentos.
- Si poseen caracteres no alfanuméricos se los reemplazo por un guion bajo (_)
- Se colapsan múltiples guines bajos (_) en uno.
- Se conserva la extensión original.

In [36]:
archivos_copiados = copiar_png_normalizados_por_ubicacion(
    ruta_origen=PATHS.DATA_ORIGINALES,
    ruta_destino=PATHS.DATA_CORREGIDOS,
)

print(f"Se copiaron {len(archivos_copiados)} archivos con los nombres normalizados.")

Se copiaron 1242 archivos con los nombres normalizados.


## Normalización de identificadores de archivos

Creación de un nuevo archivo de metadata con identificadores normalizados

Se aplican las siguentes reglas a los nombres de los archivos:
- Deben estar escritos en minúsculas.
- No deben tener acentos.
- No debe haber caracteres no alfanuméricos -> _
- Se colapsan múltiples guines bajos (_) en uno.
- Se conserva la extensión original.

In [37]:
df = cargar_metadata(PATHS.DATA_ORIGINALES)
df = normalizar_identificadores(df)
display(df.head())

,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia
0,al_03a,No,II,1,B,2,2,11.0,182,54,53,ibero
1,al_03b,No,II,1,B,2,2,11.0,178,57,46,ibero
2,al_03c,No,II,1,B,2,2,11.0,185,56,49,ibero
3,al_03d,No,II,1,B,2,2,11.0,160,41,45,ibero
4,al_03e,No,II,1,B,2,2,11.0,164,54,44,ibero


In [38]:
nombres_archivos_png = get_nombres_archivos_perfiles(PATHS.DATA_CORREGIDOS)
df = agregar_marca_identificador_en_fs(df, nombres_archivos_png)

cantidad_encontrados = contar_identificadores_encontrados(df)
cantidad_no_encontrados = contar_identificadores_no_encontrados(df)

print(
    f"Cantidad de Identificadores que NO aparecen en los nombres de los archivos: {cantidad_no_encontrados}\n"
)

Cantidad de Identificadores que NO aparecen en los nombres de los archivos: 2



In [39]:
ids_no_encontrados_ordenados = get_identificadores_no_encontrados(df)
cantidad_no_encontrados = contar_identificadores_unicos_no_encontrados(df)

print(
    "Cant. de Identificadores que aparecen en el metadata que no se corresponde con el nombre de un archivo PNG:",
    cantidad_no_encontrados,
)
print("Detalle de identificadores no encontrados en archivos PNG:")
for identificador in ids_no_encontrados_ordenados:
    print(identificador)

Cant. de Identificadores que aparecen en el metadata que no se corresponde con el nombre de un archivo PNG: 2
Detalle de identificadores no encontrados en archivos PNG:
bo_3_2_1
cc_0_8_2


In [40]:
# Archivos PNG sin registro en dataset

ids_en_df = get_identificadores_metadata(df)

archivos_sin_registro = get_archivos_sin_registro(nombres_archivos_png, ids_en_df)

cantidad_archivos_sin_registro = len(archivos_sin_registro)

print(
    f"Cantidad de archivos PNG sin registro en metadata: {cantidad_archivos_sin_registro}"
)
print("Detalle de archivos PNG no encontrados en metadata.csv:")

for archivo in archivos_sin_registro:
    print(archivo)

Cantidad de archivos PNG sin registro en metadata: 2
Detalle de archivos PNG no encontrados en metadata.csv:
bo_32_1
cc_08_2


Dado que no fueron encontrados archivos PNG correspondiente a los identificadores del archivo metadata.csv:
* bo_3_2_1
* cc_0_8_2

Y que no tienen su correspondiente registro en el archivo metadata.csv los archivos:

* bo_32_1
* cc_08_2

Por la similitud de los nombres se asume que hay una correspondencia entre ellos.

Dando por válidos los nombres de los archivos se procede a renombrar los identificadores:

* bo_3_2_1 -> bo_32_1
* cc_0_8_2 -> cc_08_2


In [41]:
correcciones = {
    "bo_3_2_1": "bo_32_1",
    "cc_0_8_2": "cc_08_2",
}

df = corregir_identificadores(df, correcciones)

print("Identificadores corregidos y metadata guardada.")
df.head()

Identificadores corregidos y metadata guardada.


,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia,id_encontrado_en_fs
0,al_03a,No,II,1,B,2,2,11.0,182,54,53,ibero,True
1,al_03b,No,II,1,B,2,2,11.0,178,57,46,ibero,True
2,al_03c,No,II,1,B,2,2,11.0,185,56,49,ibero,True
3,al_03d,No,II,1,B,2,2,11.0,160,41,45,ibero,True
4,al_03e,No,II,1,B,2,2,11.0,164,54,44,ibero,True


In [42]:
guardar_metadata(df, PATHS.DATA_CORREGIDOS)